**主要变量说明**  
| 变量名 | 含义描述 | 来源 | 数据格式 |
|------|----------|------|----------|
| Event | 某次实验完整的事件字典 | pkl 文件读入（来自 `Process.ipynb`） | `dict{event1, event2, ...}` |
| event | Event 中的某一类具体事件 | `Event['keyword']` | `dict{"Time_S": [ts1,ts3,...], "Time_E": [ts2,ts4,...]}` |
| eve_list | Event 中所有事件按时间顺序整理后的列表 | `behave_ana.sort_events()` | `list[(ts1, event), (ts2, event), ...]` |
| trial_sequence | 每一次尝试具体的结果信息 | `behave_ana.transform_sequence()` | `list[(is_correct, correct_side,reward_or_not),...]` |

_*ts指 timestamp_

base内核

In [1]:
import pickle
import behave_ana
import pandas as pd
import os
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
from scipy.stats import zscore,entropy
from scipy.signal import find_peaks
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass


**查找文件**  
找到指定路径下的pkl文件

In [2]:
import re
from pathlib import Path
from datetime import datetime

# 数据源：merged_pkls/ 含 4 个合并后的鼠×天（原 .pkl 已改名 .pkl.bak），
# 与 /mnt/HDD1/zyyk78 互补且不重复 —— 两处都扫会重复计数。
path = "/home/jyy/Documents/Beh_Ana_J/merged_pkls/"
path_extra = "/mnt/HDD1/zyyk78/"
path = Path(path)
pkl_files = list(path.rglob("*.pkl")) + list(Path(path_extra).rglob("*.pkl"))
start = datetime.strptime("20200909", "%Y%m%d")
end = datetime.strptime("20270909", "%Y%m%d")

pkl_files.sort(key=lambda f: re.search(r'(\d{8})', f.name).group(1))

filtered = []
for file in pkl_files:
    match = re.search(r'(\d{8})', file.name)
    if match:
        file_date = datetime.strptime(match.group(1), "%Y%m%d")
        if start <= file_date <= end:
            filtered.append(file)

# for file in filtered:
#     print(file)

# for file in filtered:
#     print(f"    ['{file}'],")


**数据导入和基础统计**  
把pkl路径弄到excel里面,然后把想分析的那一堆copy下就行了

In [3]:
# 自动将上面筛选出的 pkl 路径按「鼠+日期」分组：
# 同一只鼠同一天可能因中断产生多个 pkl，分组后便于后续过滤空段 + 合并连续段。
# 键 = (父目录名(鼠), 文件名前8位(日期))；值 = 该组所有 pkl 路径（按 seq 排序）。
import re as _re
from collections import defaultdict as _dd

_grp = _dd(list)
for f in filtered:
    name = f.name
    m = _re.search(r'(\d{8})', name)
    date = m.group(1) if m else '00000000'
    mouse = f.parent.name
    seq = _re.search(r'_(\d{2})\.pkl$', name)
    seq = seq.group(1) if seq else '00'
    _grp[(mouse, date)].append((seq, str(f)))

# 每组按 seq 排序，matches 变成「每组一个子列表」
matches = []
for (mouse, date), items in _grp.items():
    items.sort(key=lambda x: x[0])
    matches.append([p for _, p in items])

# 报告多 pkl 组
multi = [(k, v) for k, v in _grp.items() if len(v) > 1]
print(f'共 {len(matches)} 组（鼠×天），来自 {sum(len(v) for v in _grp.values())} 个 pkl')
print(f'同鼠同天多 pkl 的组: {len(multi)}')
for (mouse, date), items in multi:
    print(f'  {mouse} {date}: {[os.path.basename(p) for _, p in items]}')


共 893 组（鼠×天），来自 899 个 pkl
同鼠同天多 pkl 的组: 6
  D04 20260321: ['20260321_T9F_00.pkl', '20260321_T9F_01.pkl']
  D06 20260321: ['20260321_T9F_00.pkl', '20260321_T9F_01.pkl']
  D05 20260328: ['20260328_T9F_00.pkl', '20260328_T9F_01.pkl']
  E01 20260414: ['20260414_T9F_00.pkl', '20260414_T9F_01.pkl']
  C04 20260525: ['20260525_F9F_00.pkl', '20260525_F9F_01.pkl']
  C06 20260607: ['20260607_T9F_00.pkl', '20260607_T9F_01.pkl']


In [4]:
#new
import sys as _sys, os as _os
_sys.path.insert(0, _os.getcwd())   # 保证能 import 到同目录的 pkl_health
import pkl_health

# 空段阈值：单段 trial 数或 reward 数 <= 此值视为空段/废段，丢弃不合并
EMPTY_THRESH = 10
# 安全取 Event[key]['Time_S']：ModL/ModR 等可能是 numpy.array（历史脏 pkl），不能直接 ['Time_S']
# 对 array 用字符串索引会触发 IndexError: ... are valid indices
def _time_s(Event, key):
    v = Event.get(key)
    if isinstance(v, dict):
        ts = v.get('Time_S')
        return list(ts) if ts is not None else []
    return []  # array / list / None / 缺失 一律当空

# 合并同鼠同天的多段 pkl：各段时间戳独立从 ~0 起，按段偏移累加（前段 max_ts + GAP），
# 保证合并后时间单调递增、sort_events 顺序正确、trial/reward 序列连续。
MERGE_GAP = 50000  # 1s @50kHz，段间分隔，不影响归一化指标
def _all_ts_max(Event):
    mx = 0
    for k in Event:
        if isinstance(k, str) and isinstance(Event[k], dict):
            for fld in ('Time_S', 'Time_E'):
                arr = Event[k].get(fld)
                if arr:
                    mx = max(mx, max(arr))
    return mx
def _merge_events(events):
    merged = {}
    offset = 0
    for Ev in events:
        seg_max = _all_ts_max(Ev)
        for k, v in Ev.items():
            if not isinstance(k, str) or not isinstance(v, dict):
                continue  # 跳过脏 key
            if k not in merged:
                merged[k] = {'Time_S': [], 'Time_E': []}
            ts = v.get('Time_S') or []
            te = v.get('Time_E') or []
            merged[k]['Time_S'].extend([t + offset for t in ts])
            merged[k]['Time_E'].extend([t + offset for t in te])
        offset += seg_max + MERGE_GAP
    return merged

# 过滤阈值：单段 trial 数或 reward 数 <= 此值视为空段/废段，丢弃不合并
EMPTY_THRESH = 10

data_collect = []
dirty_pkls = []   # 结构异常的 pkl（某 key 不是 dict）
broken_pkls = []  # 时间轴损坏的 pkl（NaN / 常数时间戳），会污染 trial 序列
warned_pkls = []  # 有瑕疵但仍可用的 pkl（如缺 Mod 引导事件），仅提示不剔除
skipped_pkls = [] # 整体跳过的 pkl
merged_groups = []  # 发生合并的组（报告用）
dropped_segments = []  # 被过滤掉的空段（报告用）

long_event=['MinT','Blackhole']
ignore_list=['MinS','CamT','LickL','LickR','tMod']

for group_paths in matches:
    # group_paths: 同鼠同天一组的 pkl 路径列表（已按 seq 排序）
    segments = []          # 通过脏检查的 (Event, path) 列表
    group_label = os.path.basename(group_paths[0])[:8] if group_paths else '?'
    group_no_mod = False   # 本组是否缺 Mod 引导事件（进 CSV 的 ExcludeNoMod 列）

    for gp in group_paths:
        try:
            with open(gp, "rb") as pkl_f:
                Event = pickle.load(pkl_f)
        except Exception as e:
            print(f'💥 {gp} 加载失败: {type(e).__name__}: {e}')
            skipped_pkls.append(gp)
            continue

        # ====== 顶层类型检查：必须是 dict（Miniscope 钙成像 pkl 顶层是 ndarray，非行为数据）======
        if not isinstance(Event, dict):
            print(f'⚠️ 非 dict 顶层: {gp} type={type(Event).__name__}，跳过')
            skipped_pkls.append(gp)
            continue
        # ====== 脏 pkl 检查（某 key 不是 dict）======
        need_keys = ['ModL','ModR','LickL','LickR','SlnL','SlnR','ROmL','ROmR']
        bad_keys = [k for k in need_keys if Event.get(k) is not None and not isinstance(Event[k], dict)]
        if bad_keys:
            bad_types = {k: type(Event[k]).__name__ for k in bad_keys}
            print(f'⚠️ 脏 pkl: {gp}  bad_keys={bad_keys}  types={bad_types}')
            dirty_pkls.append({'file': gp, 'bad_keys': bad_keys, 'types': bad_types})
            skipped_pkls.append(gp)
            continue

        # ====== 时间轴健康检查（pkl_health 统一判据）======
        # 坏段（NaN 时间戳 / 所有 trial 压在同一时刻）若被并进 Event，
        # 会把 trial 序列与排序彻底带偏，必须在这里拦掉；
        # 空段（trial/reward 太少）留给后面的 EMPTY_THRESH 过滤统一处理。
        _h = pkl_health.check_event(Event, empty_threshold=EMPTY_THRESH)
        if _h['level'] == pkl_health.BROKEN:
            print(f'🚫 坏段(时间轴损坏) {os.path.basename(gp)}: {_h["reason"]}')
            broken_pkls.append({'file': gp, 'reason': _h['reason'], 'detail': _h['detail']})
            skipped_pkls.append(gp)
            continue
        if _h['level'] == pkl_health.WARN:
            # 有瑕疵但仍可用（例如缺 Mod 引导事件）：保留，只提示（不改变原有数据集口径）
            print(f'⚠️ 可疑段 {os.path.basename(gp)}: {_h["reason"]}')
            warned_pkls.append({'file': gp, 'reason': _h['reason'], 'detail': _h['detail']})
            if '无 Mod' in _h['reason']:
                group_no_mod = True

        segments.append((Event, gp))

    if not segments:
        print(f'⏭️ 组 {group_label} 无可用段，整体跳过')
        continue

    # ====== 空段过滤：trial/reward <= EMPTY_THRESH 丢弃 ======
    kept = []
    for Ev, gp in segments:
        n_rew = len(_time_s(Ev, 'SlnL')) + len(_time_s(Ev, 'SlnR'))
        eve_tmp = behave_ana.sort_events(Ev, ignore_list=ignore_list, long_event=long_event)
        n_trial = len(behave_ana.transform_sequence(eve_tmp))
        if n_trial <= EMPTY_THRESH or n_rew <= EMPTY_THRESH:
            print(f'🗑️ 丢弃空段 {os.path.basename(gp)}: trial={n_trial} reward={n_rew} (<= {EMPTY_THRESH})')
            dropped_segments.append({'file': gp, 'trial': n_trial, 'reward': n_rew})
        else:
            kept.append((Ev, gp))

    if not kept:
        print(f'⏭️ 组 {group_label} 过滤后无有效段，整体跳过')
        continue

    # ====== 合并：多于一段才合并，单段直接用 ======
    if len(kept) > 1:
        Event = _merge_events([Ev for Ev, _ in kept])
        merged_groups.append({
            'label': group_label,
            'parts': [os.path.basename(gp) for _, gp in kept],
            'reward': len(_time_s(Event, 'SlnL')) + len(_time_s(Event, 'SlnR')),
        })
        print(f'🔗 合并组 {group_label}: {len(kept)} 段 → 1 个 Event '
              f'(parts={[os.path.basename(gp) for _, gp in kept]})')
    else:
        Event = kept[0][0]

    pkl_path_repr = ','.join(gp for _, gp in kept)  # CSV 里记全部段路径
    date_info = os.path.basename(kept[0][1])
    date_info = str.split(date_info, '_')[0]
    date_info = date_info[:8]

    eve_list = behave_ana.sort_events(Event, ignore_list=ignore_list, long_event=long_event)

    # ====== 420 reward 截断 ======
    # reward = Sln 事件（SlnL/SlnR），与 transform_sequence 的 reward_or_not=True 一致。
    # 若某 session 的 reward 数超过 REWARD_CAP，只保留「第 REWARD_CAP 个 reward 及之前」的所有事件，
    # 并同步截断 Event 里被 reward_rate / licking_rate 直读的 Time_S 列表，保证下游所有指标口径一致。
    REWARD_CAP = 420
    reward_eves = [(t, e) for t, e in eve_list if e in ('SlnL', 'SlnR')]
    if len(reward_eves) > REWARD_CAP:
        cap_t = reward_eves[REWARD_CAP - 1][0]  # 第 420 个 reward 的时间戳（含本数）
        eve_list = [(t, e) for t, e in eve_list if t <= cap_t]
        for _rk in ('SlnL', 'SlnR', 'ROmL', 'ROmR', 'LickL', 'LickR'):
            _v = Event.get(_rk)
            if isinstance(_v, dict) and _v.get('Time_S') is not None:
                _v['Time_S'] = [ts for ts in _v['Time_S'] if ts <= cap_t]
        print(f'✂️ {group_label} reward={len(reward_eves)} > {REWARD_CAP}，截断至 cap_t={cap_t}（保留 reward={sum(1 for t,e in eve_list if e in ("SlnL","SlnR"))}）')

    tot_rew = tca_auc = log_lr = tot_acc = lrn_auc = avg_licks_after_reward = avg_licks_after_noreward = 0

    try:
        _,_,tot_rew=behave_ana.reward_rate(Event)
        tca_auc = behave_ana.cumulative_attempt_auc(eve_list)
        log_lr = behave_ana.log_lr(eve_list)
    except Exception as e:
        print(f'⚠️ {group_label} 指标(reward/auc/lr)失败: {type(e).__name__}: {e}')

    # 关键修复：用 _time_s 安全取值，不再硬 Event['ModL']['Time_S']
    if len(_time_s(Event,'ModL'))==0 and len(_time_s(Event,'ModR'))==0:
        tot_acc=0
    else:
        _,_,tot_acc=behave_ana.accuracy_rate(eve_list)
        trial_sequence = behave_ana.transform_sequence(eve_list)
        lrn_auc = behave_ana.learning_auc(trial_sequence)
        try:
            avg_licks_after_reward, avg_licks_after_noreward, total_lick_events = behave_ana.licking_rate(Event, eve_list)
        except Exception as e:
            print(f'⚠️ {group_label} licking_rate 失败: {type(e).__name__}: {e}')
            avg_licks_after_reward = avg_licks_after_noreward = 0

    all_data={
        'no_mod': group_no_mod,          # 本组是否缺 Mod（ExcludeNoMod 列）
        'pkl_path':pkl_path_repr,#原始 pkl 路径（合并组记全部段，逗号分隔）
        'date':date_info,#Date
        'tot_rew':tot_rew,#Reward Ratio
        'tot_acc':tot_acc,#Accuracy
        'lrn_auc' : lrn_auc,#Learning_auc
        'tca_auc' :tca_auc,#Cumulative_Attempt_auc
        'log_lr' : log_lr,#log_lr
        'eve_list':eve_list,
        'avg_licks_after_reward':avg_licks_after_reward,
        'avg_licks_after_noreward':avg_licks_after_noreward
    }

    data_collect.append(all_data)

# ====== 汇总 ======
print(f'\n{"=" * 60}')
print(f'脏 pkl：{len(dirty_pkls)} 个（含非 dict 事件 key，已剔除）')
print(f'坏段：{len(broken_pkls)} 个（NaN / 常数时间戳，已剔除）')
print(f'可疑段：{len(warned_pkls)} 个（仍计入数据集，仅提示）')
print(f'丢弃空段：{len(dropped_segments)} 个（trial/reward <= {EMPTY_THRESH}）')
print(f'合并组：{len(merged_groups)} 个（同鼠同天多段 → 单 Event）')
print(f'跳过 pkl 总计：{len(skipped_pkls)} 个')
print(f'成功 session：{len(data_collect)} 个（进 data_collect）')
print(f'{"=" * 60}')
if dropped_segments:
    print('丢弃空段明细：')
    for d in dropped_segments:
        print(f'  - {os.path.basename(d["file"])}: trial={d["trial"]} reward={d["reward"]}')
if merged_groups:
    print('合并组明细：')
    for m in merged_groups:
        print(f'  - {m["label"]}: {m["parts"]} → reward={m["reward"]}')
if warned_pkls:
    print('可疑段明细（仍计入数据集）：')
    for d in warned_pkls:
        print(f'  - {os.path.basename(d["file"])}: {d["reason"]}')
if broken_pkls:
    print('坏段明细：')
    for d in broken_pkls:
        print(f'  - {os.path.basename(d["file"])}: {d["reason"]}')
if dirty_pkls:
    print('脏 pkl 明细：')
    for d in dirty_pkls:
        print(f'  - {d["file"]}: bad_keys={d["bad_keys"]} types={d["types"]}')
    print('\n建议：去源头（fixing.py / ParseEvent.py）查这些 key 是怎么塞进去的，'
          '或者直接重新解析对应 evt 覆盖该 pkl。')
elif not broken_pkls:
    print('全部 pkl 干净 ✅')


⚠️ 可疑段 20250311_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250311_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250311_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250311_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250313_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250313_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250313_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
🗑️ 丢弃空段 20250313_00.pkl: trial=8 reward=414 (<= 10)
⏭️ 组 20250313 过滤后无有效段，整体跳过
⚠️ 可疑段 20250313_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250314_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250314_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250314_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
⚠️ 可疑段 20250314_00.pkl: 无 Mod 引导事件（trial 正确侧将默认按 L 处理，accuracy 会置 0）
🗑️ 丢弃空段 20250317_00.pkl: trial=1 reward=202 (<= 10)
⏭️ 组 20250317 过滤后无有效段，整体跳过
✂️ 20250317 re

Date	
Reward Ratio
Accuracy
Cumulative_Attempt_auc
Learning_auc	
log_lr	
avg_licks_after_reward
avg_licks_after_noreward

In [5]:
#两项简单的统计参数:tot_rew,tot_acc等参数,需要看哪个就打印出来
for i in ['date', 'tot_rew', 'tot_acc', 'tca_auc', 'lrn_auc', 'log_lr', 'avg_licks_after_reward', 'avg_licks_after_noreward']:
    print(f"--- {i} ---")
    for data in data_collect:
        print(f"{data[i]}")

#如果报错，可能是同一session有2个以上pkl。酌情删除（）

--- date ---
20250311
20250311
20250311
20250311
20250313
20250313
20250313
20250314
20250314
20250314
20250314
20250317
20250317
20250317
20250318
20250318
20250318
20250318
20250319
20250319
20250319
20250319
20250319
20250319
20250319
20250320
20250320
20250320
20250320
20250320
20250320
20250321
20250321
20250321
20250324
20250324
20250324
20250325
20250325
20250325
20250325
20250325
20250325
20250325
20250326
20250326
20250326
20250326
20250327
20250327
20250327
20250327
20250328
20250328
20250328
20250427
20250429
20250429
20250429
20250430
20250430
20250430
20250501
20250502
20250502
20250502
20250503
20250503
20250503
20250504
20250504
20250504
20250505
20250505
20250505
20250506
20250506
20250506
20250507
20250507
20250507
20250715
20250715
20250716
20250716
20250717
20250717
20250718
20250719
20250719
20250720
20250720
20250721
20250725
20250725
20250726
20250726
20250726
20250731
20250731
20250731
20250801
20250801
20250801
20250802
20250802
20250803
20250803
20250803
202508

Switching Latency
Incorrect Switch
MeanRewardInterval_s

In [6]:
# ⚠️ 注意：A_lick_rate（CSV 列名 MeanRewardInterval_s）的真实定义是
#   (最后一个 reward/omission 事件时间 - 第一个) / (事件数 x 50000)
# 即「reward/omission 事件的平均间隔（秒）」，与 lick 无关。
# ====== Winsorize 上限（物理合理值，防极端 session 劫持统计） ======
# 超上限的值压到上限（winsorize，不删除行）；受压行数打印在汇总里。
# 取值依据：Switching Latency 3*IQR≈37，取整 100 为保守上限（只压真正的离谱值）；
#           Incorrect Switch 同理 30；MeanRewardInterval_s 30s。
SWITCH_CAP      = 100   # 平均切换延迟上限（trial 数）
INCORRECT_CAP   = 30    # 平均错误切换次数上限
MEANRI_CAP      = 30.0  # reward/omission 平均间隔上限（秒）

A_sl=[]#平均Switching Latency (winsorized)
A_err=[]#Incorrect Switch (winsorized)
A_lick_rate=[]#MeanRewardInterval_s = reward/omission 平均间隔(秒) (winsorized)
_winsor_log=[]  # (file, 原值, 压后值) 记录被压的行
for data in data_collect:
    eve_list=data['eve_list']
    _fname=os.path.basename(str(data.get('pkl_path','')).split(',')[0])
    lat_data = behave_ana.get_switching_latency(eve_list)
    if len(lat_data) > 0 :
        _sl = sum([i[0] for i in lat_data])/len(lat_data)
        _err = sum([(len(i)-1)/2 for i in lat_data])/len(lat_data)
        lick_event = [ i for i in eve_list if i[1] in ['ROmL','ROmR','SlnL','SlnR'] ]
        _mri = (lick_event[-1][0] - lick_event[0][0]) /  (len(lick_event) * 50000)
    else:
        _sl = _err = _mri = 0
    # winsorize
    if _sl > SWITCH_CAP:
        _winsor_log.append((_fname, f'SwitchLatency {_sl:.1f}->{SWITCH_CAP}')); _sl = SWITCH_CAP
    if _err > INCORRECT_CAP:
        _winsor_log.append((_fname, f'IncorrectSwitch {_err:.1f}->{INCORRECT_CAP}')); _err = INCORRECT_CAP
    if _mri > MEANRI_CAP:
        _winsor_log.append((_fname, f'MeanRI {_mri:.1f}->{MEANRI_CAP}')); _mri = MEANRI_CAP
    A_sl.append(_sl); A_err.append(_err); A_lick_rate.append(_mri)

print(f"[Winsorize] caps: SwitchLatency>{SWITCH_CAP}, IncorrectSwitch>{INCORRECT_CAP}, MeanRI>{MEANRI_CAP}")
if _winsor_log:
    for f,info in _winsor_log:
        print(f"    · {f:32s} {info}")
    print(f"    total winsorized rows: {len(_winsor_log)}")
else:
    print("    none")

In [7]:
data_collect2= [A_sl,A_err,A_lick_rate]
# for i in data_collect2:
    # for data in i:
        # print(data)
    # print(f"--- ---")


将所有指标保存为CSV表格


In [8]:
# 将所有指标保存为CSV表格
import csv

# 注意：data_collect 可能比 matches 短（跳过了脏 pkl），
# A_sl/A_err/A_lick_rate 是按 data_collect 长度生成的，
# 所以必须按 data_collect 遍历，用同一个 i 取 A_*，不能再用 matches 的 index。
# ===== 口径开关 =====
# 缺 Mod 引导事件的 session：transform_sequence 会把正确侧默认成 'L'，
# 由此得到的 Accuracy/Learning_auc 是伪值。IncludeNoMod=True 时照旧计入（列 ExcludeNoMod 标记）；
# False 时从 CSV 中剔除这些 session。
IncludeNoMod = True

rows = []
for i, data in enumerate(data_collect):
    _parts = str(data.get('pkl_path','')).split(',')
    rows.append({
        'Train_file': data.get('pkl_path', ''),
        'Mouse_ID': os.path.basename(os.path.dirname(_parts[0])),
        'n_segments': len(_parts),
        'Merged': any('_merged' in os.path.basename(x) for x in _parts),
        'ExcludeNoMod': bool(data.get('no_mod', False)),
        'Date': data['date'],
        'Reward Ratio': data['tot_rew'],
        'Accuracy': data['tot_acc'],
        'Cumulative_Attempt_auc': data['tca_auc'],
        'Learning_auc': data['lrn_auc'],
        'log_lr': data['log_lr'],
        'avg_licks_after_reward': data['avg_licks_after_reward'],
        'avg_licks_after_noreward': data['avg_licks_after_noreward'],
        'Switching Latency': A_sl[i],
        'Incorrect Switch': A_err[i],
        'MeanRewardInterval_s': A_lick_rate[i],
    })

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_dir = "/home/jyy/Documents/Beh_Ana_J"
csv_path = os.path.join(csv_dir, f"analysis_results_{timestamp}.csv")
fieldnames = [
    'Train_file', 'Mouse_ID', 'n_segments', 'Merged', 'ExcludeNoMod', 'Date',
    'Reward Ratio', 'Accuracy',
    'Cumulative_Attempt_auc', 'Learning_auc', 'log_lr',
    'avg_licks_after_reward', 'avg_licks_after_noreward',
    'Switching Latency', 'Incorrect Switch', 'MeanRewardInterval_s'
]
_rows_out = rows if IncludeNoMod else [r for r in rows if not r['ExcludeNoMod']]
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in _rows_out:
        writer.writerow(row)

print(f"CSV saved to: {csv_path}")
print(f"Total rows written: {len(_rows_out)} (IncludeNoMod={IncludeNoMod}, 其中缺Mod剔除={len(rows)-len(_rows_out)})")
print(f"（matches={len(matches)}, 成功={len(data_collect)}, 跳过={len(matches)-len(data_collect)}）")

CSV saved to: /home/jyy/Documents/Beh_Ana_J/analysis_results_20261027_170159.csv
Total rows written: 865
（matches=893, 成功=865, 跳过=28）


In [9]:
# === 诊断：为什么这些指标是 0？===
# 注意：Ev['ModL'] 可能是 array（脏 pkl），用 _time_s 安全取值
def _time_s_d(Event, key):
    v = Event.get(key)
    if isinstance(v, dict):
        ts = v.get('Time_S')
        return list(ts) if ts is not None else []
    return []

diag = []
for pkl_path in matches:
    with open(pkl_path[0], "rb") as f:
        Ev = pickle.load(f)
    modL_empty = (len(_time_s_d(Ev,'ModL')) == 0)
    modR_empty = (len(_time_s_d(Ev,'ModR')) == 0)
    skip_else_branch = modL_empty and modR_empty  # 这是 data_collect 里 if 的命中条件

    # 重跑 get_switching_latency 看为什么返回空
    long_event=['MinT','Blackhole']
    ignore_list=['MinS','CamT','LickL','LickR','tMod']
    eve_list_d = behave_ana.sort_events(Ev, ignore_list=ignore_list, long_event=long_event)
    lat_d = behave_ana.get_switching_latency(eve_list_d)

    # Sln/ROm 这几个事件数量，决定 A_lick_rate 能不能算
    lick_event = [i for i in eve_list_d if i[1] in ['ROmL','ROmR','SlnL','SlnR']]

    diag.append({
        'file': os.path.basename(pkl_path[0]),
        'ModL_empty': modL_empty,
        'ModR_empty': modR_empty,
        'if_branch_triggered(=跳过else)': skip_else_branch,
        'SlnL_n': len(_time_s_d(Ev,'SlnL')),
        'SlnR_n': len(_time_s_d(Ev,'SlnR')),
        'ROmL_n': len(_time_s_d(Ev,'ROmL')),
        'ROmR_n': len(_time_s_d(Ev,'ROmR')),
        'eve_list_total': len(eve_list_d),
        'switch_lat_data_len': len(lat_d),
        'lick_event_n': len(lick_event),
    })

pd.DataFrame(diag)

AttributeError: 'numpy.ndarray' object has no attribute 'get'

其他功能

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 从剪贴板读取Excel复制的数据（自动检测分隔符）
df = pd.read_clipboard(header=None)  
# 转换为列表（每行一个子列表）
data_list = df.values.tolist()  
data_0 = data_list[0]
with open(data_0[0],"rb") as pkl_f:
    Event=pickle.load(pkl_f)

fig, (ax1, ax1_5, ax2) = plt.subplots(3, 1, figsize=(20, 5), 
                                   gridspec_kw={'height_ratios': [2 , 1 , 2]},sharex=True,)
sns.rugplot(Event['SlnL']['Time_S'] , ax=ax1, height=0.8, linewidth=0.5, color='green',alpha=0.5)
sns.rugplot(Event['SlnR']['Time_S'] , ax=ax2, height=0.8, linewidth=0.5, color='green',alpha=0.5)
sns.rugplot(Event['ROmL']['Time_S'] , ax=ax1, height=0.8, linewidth=0.5, color='red',alpha=0.5)
sns.rugplot(Event['ROmR']['Time_S'] , ax=ax2, height=0.8, linewidth=0.5, color='red',alpha=0.5)
sns.rugplot(Event['ModL']['Time_S'] , ax=ax1, height=1, linewidth=1.5, color='blue')
sns.rugplot(Event['ModR']['Time_S'] , ax=ax2, height=1, linewidth=1.5, color='blue')

if 'Blackhole' in Event:
    for i in range(len(Event['Blackhole']['Time_S'])):
        ax1_5.fill_betweenx(y=[-0.5, 0.5], x1 = Event['Blackhole']['Time_S'][i], x2=Event['Blackhole']['Time_E'][i],color='red')
ax1.set_yticks([])
ax1.set_ylabel('Left Side')
ax2.set_yticks([])
ax2.set_ylabel('Right Side')
ax1_5.set_yticks([])
ax1_5.set_ylabel('Uncertain Area')
plt.show()

**自回归分析**  
这里只依赖前面载入的pkl文件  
统计这些pkl中选择的历史以来情况

In [ ]:
fig_bys,ax_bys=plt.subplots(figsize=(8, 5),dpi=300)
import numpy as np
para_list=[]
bias_list=[]
acc_list=[]
for i in range(1000):
    sw_seq= []
    ns_seq= []
    for data_dict in data_collect:
        sw=behave_ana.get_switch_sequence(data_dict['eve_list'],10)
        sw_seq.extend(sw)
        ns_seq.extend(behave_ana.get_random_ns_sequence(data_dict['eve_list'],10,len(sw)))
    para,bias,acc=behave_ana.switch_bayes_reg(sw_seq,ns_seq)
    para_list.append(para)
    bias_list.append(bias)
    acc_list.append(acc)
para_list=np.array(para_list)
bias_list=np.array(bias_list)
acc_list=np.array(acc_list)

means = np.mean(para_list, axis=0)[0]
stds = np.std(para_list, axis=0)[0]
means_bias = np.mean(np.squeeze(bias_list))
stds_bias = np.std(np.squeeze(bias_list))
means_acc = np.mean(np.squeeze(acc_list))
stds_acc = np.std(np.squeeze(acc_list))

x = [i for i in range(-means.shape[0],0)]

# 绘制误差线折线图len_lat_data
ax_bys.errorbar(
    [i for i in range(-means.shape[0],0)],   
    y=means, 
    yerr=stds,
    fmt='-o',         # 线型：实线+圆圈标记
    capsize=5,        # 误差线端盖长度
    capthick=2,       # 端盖粗细
    elinewidth=1,     # 误差线粗细
    label='coef ± Std, Expert'
)

ax_bys.errorbar(
    0, 
    y=means_bias, 
    yerr=stds_bias,
    fmt='-o',         # 线型：实线+圆圈标记
    capsize=5,        # 误差线端盖长度
    capthick=2,       # 端盖粗细
    elinewidth=1,     # 误差线粗细
    label='bias ± Std, Expert'
)
ax_bys.axhline(y=0, color='red', linewidth=3, linestyle='-')
print(f'Pred_Acc={means_acc:.2f}±{stds_acc:.2f}')

ax_bys.set_ylabel('Coefficient')
ax_bys.set_xlabel('Trial')
ax_bys.set_title('Logistic Autoregression Coefficient, Bootstrap=1000')
ax_bys.set_ylim(-3, 3)
legend=ax_bys.legend()
display(fig_bys)

**显著性分析比对**   
需要你copy一个表格(直接从excel中框选复制即可)  
参考格式如下:

| Tag | Reward Ratio | Accuracy | Switching Latency | Incorrect Switch | Trial Gap |
|-----|--------------|----------|-------------------|------------------|-----------|
| T8  | 0.534460338  | 0.613312203 | 6.117647059  | 1.441176471  | 3.340758352 |
| T8  | 0.482142857  | 0.18907563  | 16            | 2.357142857  | 2.369864286 |
| T8  | 0.60960961   | 0.599099099 | 9.866666667  | 1.033333333  | 1.945558799 |
| T8  | 0.56097561   | 0.561683599 | 12.73333333  | 0.933333333  | 2.555443079 |
| T9  | 0.489671932  | 0.413122722 | 13            | 1.818181818  | 3.237225881 |
| T9  | 0.363471971  | 0.121661721 | 13.25         | 4.375         | 4.492284458 |
| T9  | 0.520779221  | 0.507792208 | 4.875         | 6.6875        | 3.156573948 |
| T9  | 0.24504084   | 0.175029172 | 5.333333333  | 14.33333333  | 4.780938203 |
  
_并且接下来还有一些要对应修改的_

In [ ]:
ref_tag= 'T9'  #这个是进行显著性分析的参考项
feature_columns = ['Reward Ratio', 'Accuracy', 'Switching Latency', 'Incorrect Switch', 'MeanRewardInterval_s']
STAT_TEST = lambda x, y: stats.mannwhitneyu(x, y, alternative="two-sided")  # M-W U-Test
#STAT_TEST = lambda x, y: stats.ttest_ind(x,y)  #T-Test

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# 从剪贴板读取数据
df = pd.read_clipboard()

# 查看数据结构
print("数据前几行：")
print(df.head())
print("\n数据信息：")
print(df.info())

# 检查数据列名
print("\n数据列名：")
print(df.columns.tolist())


# 数据整理
df_melted = df.melt(id_vars=['Tags'], value_vars=feature_columns, 
                    var_name='Data_Type', value_name='Value')

# 统计检验 - 对每个数据类型，将每个组与ref_tag进行比较
test_results = {}

for data_type in feature_columns:
    data_subset = df_melted[df_melted['Data_Type'] == data_type]
    T9_data = data_subset[data_subset['Tags'] == ref_tag]['Value'].values
    
    # 获取所有唯一的标签
    other_tags = [tag for tag in data_subset['Tags'].unique() if tag != ref_tag]
    
    results = {}
    for tag in other_tags:
        tag_data = data_subset[data_subset['Tags'] == tag]['Value'].values
        if len(T9_data) > 0 and len(tag_data) > 0:
            _, p_value = STAT_TEST(T9_data, tag_data)
            results[tag] = p_value
    
    test_results[data_type] = results

print("\nT检验结果：")
for data_type, tag_results in test_results.items():
    print(f"\n{data_type}:")
    for tag, p_val in tag_results.items():
        print(f"  {tag} vs T9: p-value = {p_val:.4f}")

# 创建图形 - 5个特征，使用2x3的布局
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# 设置颜色 - 使用seaborn默认调色板
unique_tags = df['Tags'].unique()
palette = sns.color_palette("Set2", len(unique_tags))



# 为每个特征绘制箱型图
for i, feature in enumerate(feature_columns):
    if i < len(axes):
        ax = axes[i]
        step_height = 0
        # 绘制箱型图
        sns.boxplot(data=df, x='Tags', y=feature, ax=ax, palette=palette)
        sns.stripplot(data=df, x='Tags', y=feature, ax=ax, color='black', alpha=0.6, size=4, jitter=True)
        
        # 标注显著性
        if feature in test_results:
            y_max = df[feature].max()
            y_min = df[feature].min()
            y_range = y_max - y_min
            
            # 为每个非T9标签添加显著性标注
            x_positions = {}
            for j, tag in enumerate(unique_tags):
                x_positions[tag] = j
            
            T9_position = x_positions[ref_tag]
            
            for tag, p_value in test_results[feature].items():
                if tag in x_positions:
                    tag_position = x_positions[tag]
                    mid_point = (T9_position + tag_position) / 2
                    
                    # 计算标注位置
                    significance_y = y_max + 0.1 * y_range + step_height*0.1*y_range
                    
                    # 添加显著性符号
                    if p_value < 0.001:
                        symbol = '***'
                    elif p_value < 0.01:
                        symbol = '**'
                    elif p_value < 0.05:
                        symbol = '*'
                    else:
                        symbol = 'ns'
                    
                    # 添加横线和标注
                    ax.plot([T9_position, tag_position], [significance_y , significance_y ], 'k-', lw=1)
                    ax.plot([T9_position, T9_position], [significance_y-0.01*y_range , significance_y ], 'k-', lw=1)
                    ax.plot([tag_position, tag_position], [significance_y-0.01*y_range , significance_y ], 'k-', lw=1)
                    ax.text(mid_point, significance_y + 0.02*y_range , symbol, 
                           ha='center', va='bottom', fontsize=10, fontweight='bold')
                step_height += 1      
        
        ax.set_title(f'{feature} Distribution', fontsize=14, fontweight='bold')
        ax.set_ylabel(feature, fontsize=12)
        ax.set_xlabel('Tags', fontsize=12)
        
        # 旋转x轴标签以避免重叠
        ax.tick_params(axis='x', rotation=45)

# 如果子图数量多于特征数量，隐藏多余的子图
for i in range(len(feature_columns), len(axes)):
    axes[i].set_visible(False)

# 调整布局
plt.tight_layout()

# 显示图形
plt.show()

# 显示描述性统计
print("\n描述性统计：")
desc_stats = df.groupby('Tags')[feature_columns].describe()
print(desc_stats)

# 创建显著性结果表格
print("\n显著性总结（与T9比较）:")
significance_summary = []
for feature in feature_columns:
    if feature in test_results:
        for tag, p_val in test_results[feature].items():
            if p_val < 0.001:
                sig_level = '***'
            elif p_val < 0.01:
                sig_level = '**'
            elif p_val < 0.05:
                sig_level = '*'
            else:
                sig_level = 'ns'
            significance_summary.append({
                'Feature': feature,
                'Comparison': f'{tag} vs T9',
                'p-value': f'{p_val:.4f}',
                'Significance': sig_level
            })

significance_df = pd.DataFrame(significance_summary)
print(significance_df)